# Assemble county control totals

## Introduction

For the purposes of demographic modeling for the Metropolitan Transportation Plan (MTP), future projections of several variables are produced at the county level.  Collectively these projections are known as "county control totals" or "county controls" because they are used to control or constrain projections for smaller geographies. The county controls are updated every four years in conjunction with the MTP.  The included variables are:

  - Total population
  - Group quarters population
  - Household population
  - Total population by large age group (17 and under, 18 to 64, 65 and over)
  - Total population by 5-year age cohort
  - Household population
  - Household population by large age group
  - Total jobs
  - Jobs by large job category (office, industrial, retail services, retail goods, other)
  - **Households** (divide household population by average household size)
  - **Housing units** (divide households by housing unit occupancy rate)
    **Labor force** (multiply working age population by labor force participation rate)
  - **Workers** (multiply labor force by employment rate)

This script collects the variables, assembles them into a single standardized long-form table that is optimized for machine-readability.  A downstream workflow (morpc-county-controls-profile) produces an Excel spreadsheet intended for human-readability and various other deliverable data products.

Some variables are produced by complex methods.  Those are produced using separate upstream workflows.  Other variables are produced using simple ratios or scaling factors.  Those are produced by this notebook.  The variables that are produced by this notebook are those that are bolded in the above list.

This process is dependent on upstream processes. See the "Prerequisites" section below.

The workflow defined herein is identified as workflow ID #23 in the [Data Team Master Document List](https://morpc1.sharepoint.com/:x:/s/GISteam/EfC4j3HhohZCrSZzxJdyt5cBFEqVD7zHick8ZW0INqgCYA?e=0WhrAI). References to document list identifiers are denoted by a number in brackets, e.g. [23].

## Process outline

## Prerequisites and usage notes

  - Outputs of one or more upstream workflows must be available at the indicated paths. Make sure that those outputs are up to date prior to running this script. 
  - This script includes several intentional RuntimeError instances that may be triggered to alert the user to conditions that may require their attention. If the script triggers one of these errors, review the error, verify that the condition is acceptable or resolve any issues, then proceed.

## Setup

### Import required packages

In [1]:
import os
import morpc
import morpc_census
import dotenv
import logging
import pandas as pd

### Start logging

In [2]:
morpc.logs.config_logs()
logger = logging.getLogger(__name__)

2026-08-25 21:53:29,586 | INFO | morpc.logs.config_logs: Log filename not specified.  Using default filename.
2026-08-25 21:53:29,586 | INFO | morpc.logs.config_logs: Set up logging save to file "morpc-county-controls.ipynb.log", log level "info"


### User-specified parameters

In [104]:
# Because new county controls are produced every four years, we'll assign a vintage identifier to the outputs to differentiate 
# them from outputs produced in other vintages.  The vintage is captured as a year.  Specify the four-digit integer year 
# in VINTAGE_YEAR.
VINTAGE_YEAR = 2026

# Specify the ACS 5-year estimates survey year as a four-digit integer.  This is the final year in the 5-year period.
ACS_YEAR = 2024

# When STALE_DATA_INTERRUPT == True, the script will produce a RuntimeError in certain situations where the input 
# data may be stale and updates might be required prior to running the script.  Otherwise, a warning will be generated 
# but script execution will continue.  Regardless of whether an error or warning occurs, be sure to verify the readiness 
# of all input data.
STALE_DATA_INTERRUPT = True

# This script may pull data from outputs of upstream workflows.  The locations of these outputs are specified by their path relative
# a GitHub root directory. This is a single directory which is presumed to contain local working copies of MORPC GitHub repositories.
# Specify the path to the directory on your system where the local working copies are stored. By default, the GitHub root directory is
# assumed to be one level up from this script.
GITHUB_ROOT = "../"

# Specify the path to the directory where the input data is stored. Sometimes the data may be sourced from this location and sometimes 
# it may be sourced from elsewhere and archived here.
INPUT_DIR = "./input_data"

# Specify the path to the directory where the output data is stored. Typically it is not necessary to change this, and changing it for 
# established scripts may break other scripts that depend on outputs from this one.
OUTPUT_DIR = "./output_data"

# This script requires a valid Census API key to be stored in the environment variable CENSUS_API_KEY. You may define 
# the variable via any means provided by the operating system, or you may define the variable in a dotenv file at the path 
# specified in DOTENV_PATH (typically .env in the same directory as the script). In the event that the variable is defined 
# in two places, the system variable will be used.
DOTENV_PATH = "./.env"

# Set EXPORT_NOTEBOOK_AS_HTML to True to automatically export the notebook in HTML format when the script finishes.  Set to False to skip the export
EXPORT_NOTEBOOK_AS_HTML = True

### Static parameters

In [105]:
VINTAGE_FREQ = "Y-DEC"

COMMON_ID_VARS = ['vintage_period', 'vintage_period_freq', 'scenario', 'geoidfq', 'sumlevel', 'name',
                  'reference_period', 'reference_period_freq']

# Specify the ACS survey as 'acs5' (for 5-year estimates)
ACS_SURVEY = 'acs5'


### Load environment variables

This script requires a valid Census API key to be stored in the environment variable `CENSUS_API_KEY`.  You may define the variable via any means provided by the operating system, or you may define the variable in a dotenv file at the path specified in `DOTENV_PATH` (typically `.env` in the same directory as the script).  In the event that the variable is defined in two places, the system variable will be used.

In [4]:
if(dotenv.load_dotenv(dotenv_path=DOTENV_PATH, verbose=True, override=False) == True):
    logger.info("Successfully loaded environment variables.")
    # Optional check for specific variables (replace VAR_NAME with the name of the variable to check)
    if not "CENSUS_API_KEY" in os.environ.keys():
        logger.error("CENSUS_API_KEY is not defined in environment variables")
        raise RuntimeError
else:
    logger.error(f"Failed to load environment variables from {DOTENV_PATH}")
    raise RuntimeError

2026-08-25 21:53:29,605 | INFO | __main__.<module>: Successfully loaded environment variables.


### Define inputs

The following datasets are required by this notebook. They will be retrieved from the specified location and temporarily stored in INPUT_DIR.

#### Create input data directory

Create input data directory if it doesn't exist.

In [5]:
inputDir = os.path.normpath(INPUT_DIR)
if not os.path.exists(inputDir):
    os.makedirs(inputDir)

#### County projections of total population [624]

This input includes total population, household population, and group quarters population.

In [356]:
POP_TOTAL_RESOURCE_PATH = os.path.join(GITHUB_ROOT, "morpc-projections-countypop/output_data/morpc-projections-countypop.resource.yaml")
logger.info(f"Resource: {POP_TOTAL_RESOURCE_PATH}")

2026-08-26 09:49:51,089 | INFO | __main__.<module>: Resource: ../morpc-projections-countypop/output_data/morpc-projections-countypop.resource.yaml


#### County population projections by 5-year age cohort [540]

This input includes total population, household population, and group quarters population by 5-year age cohort.

In [6]:
POP_COHORT_RESOURCE_PATH = os.path.join(GITHUB_ROOT, "morpc-projections-countypop/output_data/morpc-projections-countypop-5year.resource.yaml")
logger.info(f"Resource: {POP_COHORT_RESOURCE_PATH}")

2026-08-25 21:53:29,638 | INFO | __main__.<module>: Resource: ../morpc-projections-countypop/output_data/morpc-projections-countypop-5year.resource.yaml


#### County population projections by large age group [541]

This input includes total population, household population, and group quarters population by large age group (17 and under, 18 to 64, 65 and over).

In [7]:
POP_LARGEGROUP_RESOURCE_PATH = os.path.join(GITHUB_ROOT, "morpc-projections-countypop/output_data/morpc-projections-countypop-largegroups.resource.yaml")
logger.info(f"Resource: {POP_LARGEGROUP_RESOURCE_PATH}")

2026-08-25 21:53:29,652 | INFO | __main__.<module>: Resource: ../morpc-projections-countypop/output_data/morpc-projections-countypop-largegroups.resource.yaml


#### County projections of total jobs [TBD]

In [545]:
JOBS_TOTAL_RESOURCE_PATH = os.path.join(GITHUB_ROOT, "morpc-projections-countyjobs/output_data/morpc-projections-countyjobs.resource.yaml")
logger.info(f"Resource: {JOBS_TOTAL_RESOURCE_PATH}")

2026-08-27 21:47:14,390 | INFO | __main__.<module>: Resource: ../morpc-projections-countyjobs/output_data/morpc-projections-countyjobs.resource.yaml


#### Census ACS 5-year estimates [86]

The script will retrieve data from the Census ACS 5-year estimates API using the tools in the `morpc-census` package.  More information about the API is available at https://www.census.gov/data/developers/data-sets/acs-5year.html. The raw data will be archived in the directory specified in `INPUT_DIR`.

Specifically, we'll use the following tables:
  - B25010 (Average Household Size of Occupied Housing Units by Tenure)
  - B25002 (Occupancy Status)
  - B23025 (Employment Status)

### Define outputs

#### Create output data directory

Create output data directory if it doesn't exist.

In [8]:
outputDir = os.path.normpath(OUTPUT_DIR)
if not os.path.exists(outputDir):
    os.makedirs(outputDir)   

#### County controls long-form table [TBD]

In [508]:
OUTPUT_LONG_FILENAME = "morpc-county-controls-long.csv"
OUTPUT_LONG_PATH = os.path.join(outputDir, OUTPUT_LONG_FILENAME)
OUTPUT_LONG_SCHEMA_PATH = OUTPUT_LONG_PATH.replace(".csv",".schema.yaml")
OUTPUT_LONG_RESOURCE_PATH = OUTPUT_LONG_PATH.replace(".csv",".resource.yaml")
logger.info(f"Data: {OUTPUT_LONG_PATH}")
logger.info(f"Schema: {OUTPUT_LONG_SCHEMA_PATH}")
logger.info(f"Resource: {OUTPUT_LONG_RESOURCE_PATH}")

2026-08-26 12:18:48,693 | INFO | __main__.<module>: Data: output_data\morpc-county-controls-long.csv
2026-08-26 12:18:48,695 | INFO | __main__.<module>: Schema: output_data\morpc-county-controls-long.schema.yaml
2026-08-26 12:18:48,700 | INFO | __main__.<module>: Resource: output_data\morpc-county-controls-long.resource.yaml


#### County controls wide-form table [TBD]

In [507]:
OUTPUT_WIDE_FILENAME = "morpc-county-controls-wide.csv"
OUTPUT_WIDE_PATH = os.path.join(outputDir, OUTPUT_WIDE_FILENAME)
OUTPUT_WIDE_SCHEMA_PATH = OUTPUT_WIDE_PATH.replace(".csv",".schema.yaml")
OUTPUT_WIDE_RESOURCE_PATH = OUTPUT_WIDE_PATH.replace(".csv",".resource.yaml")
logger.info(f"Data: {OUTPUT_WIDE_PATH}")
logger.info(f"Schema: {OUTPUT_WIDE_SCHEMA_PATH}")
logger.info(f"Resource: {OUTPUT_WIDE_RESOURCE_PATH}")

2026-08-26 12:18:45,721 | INFO | __main__.<module>: Data: output_data\morpc-county-controls-wide.csv
2026-08-26 12:18:45,722 | INFO | __main__.<module>: Schema: output_data\morpc-county-controls-wide.schema.yaml
2026-08-26 12:18:45,723 | INFO | __main__.<module>: Resource: output_data\morpc-county-controls-wide.resource.yaml


## Prepare input data

### Projected total population

Load and validate the data.

In [377]:
(popTotal, popTotalResource, popTotalGroupSchema) = morpc.frictionless.load_data(POP_TOTAL_RESOURCE_PATH, archiveDir=inputDir, validate=True)

2026-08-26 09:54:53,308 | INFO | morpc.frictionless.frictionless.load_data: Loading Frictionless Resource file at location ..\morpc-projections-countypop\output_data\morpc-projections-countypop.resource.yaml
2026-08-26 09:54:53,331 | INFO | morpc.frictionless.frictionless.load_data: Using schema path specified in resource file.
2026-08-26 09:54:53,335 | INFO | morpc.frictionless.frictionless.load_data: Copying data, resource file, and schema (if applicable) to directory input_data
2026-08-26 09:54:53,345 | INFO | morpc.frictionless.frictionless.load_data: --> Data file: input_data\morpc-projections-countypop.csv
2026-08-26 09:54:53,347 | INFO | morpc.frictionless.frictionless.load_data: --> Resource file: input_data\morpc-projections-countypop.resource.yaml
2026-08-26 09:54:53,349 | INFO | morpc.frictionless.frictionless.load_data: --> Schema file: input_data\morpc-projections-countypop.schema.yaml
2026-08-26 09:54:53,351 | INFO | morpc.frictionless.frictionless.load_data: Validating r

Inspect the raw data.

In [378]:
popTotal

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,POP,POPHH,POPGQ
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,242032,239707,2325
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,258927,256602,2325
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2035,271837,268942,2895
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2040,282457,278970,3487
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2045,293027,289205,3822
...,...,...,...,...,...,...,...,...,...
478,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2035,1801228,1766110,35118
479,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2040,1885583,1847669,37914
480,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2045,1961073,1921714,39359
481,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2050,2030658,1990641,40017


Rename the data variables to standard form.

In [379]:
popTotal = popTotal.rename(columns={
    "POP":"pop",
    "POPHH":"pophh",
    "POPGQ":"popgq"
})
dataVars = ["pop","pophh","popgq"]

Adjust the identifier variables to match the common schema.

In [380]:
popTotal = popTotal.rename(columns={
    "VINTAGE":"vintage_period",
    "SCENARIO":"scenario",
    "GEOIDFQ":"geoidfq",
    "SUMLEVEL":"sumlevel",
    "NAME":"name",
    "YEAR":"reference_period"
})

popTotal["vintage_period_freq"] = VINTAGE_FREQ
popTotal["reference_period_freq"] = "Y-JUN"

Put the columns in an intuitive order.

In [381]:
popTotal = popTotal.filter(items=COMMON_ID_VARS + dataVars, axis="columns")

Use the common list of ID variables as the index.

In [382]:
popTotal = popTotal.set_index(COMMON_ID_VARS)

Inspect the transformed data.

In [383]:
popTotal

pop  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   242032   
                                                                                                                     2030             Y-JUN                   258927   
                                                                                                                     2035             Y-JUN                   271837   
                                                                                                                     2040             Y-JUN                   282457   
                                                                                                                     2045             Y-JUN                   293027   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1801228   
                                                                                                                     2040             Y-JUN                  1885583   
                                                                                                                     2045             Y-JUN                  1961073   
                                                                                                                     2050             Y-JUN                  2030658   
                                                                                                                     2055             Y-JUN                  2095623   

                                                                                                                                                               pophh  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   239707   
                                                                                                                     2030             Y-JUN                   256602   
                                                                                                                     2035             Y-JUN                   268942   
                                                                                                                     2040             Y-JUN                   278970   
                                                                                                                     2045             Y-JUN                   289205   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1766110   
                                                                                                                     2040             Y-JUN                  1847669   
                                                                                                                     2045             Y-JUN                  1921714   
                                                                                                                     2050       

### Projected population by large age group

Load and validate the data.

In [368]:
(popLargeGroup, popLargeGroupResource, popLargeGroupSchema) = morpc.frictionless.load_data(POP_LARGEGROUP_RESOURCE_PATH, archiveDir=inputDir, validate=True)

2026-08-26 09:54:44,856 | INFO | morpc.frictionless.frictionless.load_data: Loading Frictionless Resource file at location ..\morpc-projections-countypop\output_data\morpc-projections-countypop-largegroups.resource.yaml
2026-08-26 09:54:44,870 | INFO | morpc.frictionless.frictionless.load_data: Using schema path specified in resource file.
2026-08-26 09:54:44,871 | INFO | morpc.frictionless.frictionless.load_data: Copying data, resource file, and schema (if applicable) to directory input_data
2026-08-26 09:54:44,878 | INFO | morpc.frictionless.frictionless.load_data: --> Data file: input_data\morpc-projections-countypop-largegroups.csv
2026-08-26 09:54:44,879 | INFO | morpc.frictionless.frictionless.load_data: --> Resource file: input_data\morpc-projections-countypop-largegroups.resource.yaml
2026-08-26 09:54:44,880 | INFO | morpc.frictionless.frictionless.load_data: --> Schema file: input_data\morpc-projections-countypop-largegroups.schema.yaml
2026-08-26 09:54:44,881 | INFO | morpc.f

Inspect the raw data.

In [369]:
popLargeGroup

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,AGEGROUP,POP,POPHH,POPGQ
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,17U,53556,53431,124
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1864,149855,148514,1342
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,65O,38621,37762,859
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,17U,52391,52266,124
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,1864,159714,158373,1342
...,...,...,...,...,...,...,...,...,...,...
1444,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2050,1864,1329105,1305278,23828
1445,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2050,65O,344105,331485,12620
1446,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,17U,361861,358101,3760
1447,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,1864,1359640,1335353,24287


Melt the data to long form.

In [370]:
popLargeGroup = popLargeGroup.melt(id_vars=['VINTAGE', 'SCENARIO', 'GEOIDFQ', 'SUMLEVEL', 'NAME', 'YEAR', 'AGEGROUP'])
popLargeGroup.head()

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,AGEGROUP,variable,value
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,17U,POP,53556
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1864,POP,149855
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,65O,POP,38621
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,17U,POP,52391
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,1864,POP,159714


Create composite variable names in standard form from the original variable names and the age group, then drop the original variable names and age group.

In [371]:
popLargeGroup["standard_var"] = (popLargeGroup["variable"].str.replace("POP","POPA") + "_" + popLargeGroup["AGEGROUP"]).str.lower()
popLargeGroup = popLargeGroup.drop(columns=["AGEGROUP","variable"])
dataVars = list(popLargeGroup["standard_var"].unique())

Pivot the table to wide form using the new variable names.

In [372]:
popLargeGroup = popLargeGroup.pivot(index=['VINTAGE', 'SCENARIO', 'GEOIDFQ', 'SUMLEVEL', 'NAME', 'YEAR'], columns="standard_var", values="value").reset_index()
popLargeGroup.columns.name = None
popLargeGroup.head()

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,popa_17u,popa_1864,popa_65o,popagq_17u,popagq_1864,popagq_65o,popahh_17u,popahh_1864,popahh_65o
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,53556,149855,38621,124,1342,859,53431,148514,37762
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,52391,159714,46822,124,1342,859,52266,158373,45963
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2035,47561,170931,53345,146,1613,1136,47414,169319,52209
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2040,43921,178493,60043,155,1878,1454,43765,176616,58589
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2045,43938,183166,65923,138,1991,1693,43800,181175,64230


Adjust the identifier variables to match the common schema.

In [373]:
popLargeGroup = popLargeGroup.rename(columns={
    "VINTAGE":"vintage_period",
    "SCENARIO":"scenario",
    "GEOIDFQ":"geoidfq",
    "SUMLEVEL":"sumlevel",
    "NAME":"name",
    "YEAR":"reference_period"
})

popLargeGroup["vintage_period_freq"] = VINTAGE_FREQ
popLargeGroup["reference_period_freq"] = "Y-JUN"

Put the columns in an intuitive order.

In [374]:
popLargeGroup = popLargeGroup.filter(items=COMMON_ID_VARS + dataVars, axis="columns")

Use the common list of ID variables as the index.

In [375]:
popLargeGroup = popLargeGroup.set_index(COMMON_ID_VARS)

Inspect the transformed data.

In [376]:
popLargeGroup

popa_17u  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq             
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                     53556   
                                                                                                                     2030             Y-JUN                     52391   
                                                                                                                     2035             Y-JUN                     47561   
                                                                                                                     2040             Y-JUN                     43921   
                                                                                                                     2045             Y-JUN                     43938   
...                                                                                                                                                               ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                    344473   
                                                                                                                     2040             Y-JUN                    346179   
                                                                                                                     2045             Y-JUN                    352049   
                                                                                                                     2050             Y-JUN                    357448   
                                                                                                                     2055             Y-JUN                    361861   

                                                                                                                                                             popa_1864  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq              
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                     149855   
                                                                                                                     2030             Y-JUN                     159714   
                                                                                                                     2035             Y-JUN                     170931   
                                                                                                                     2040             Y-JUN                     178493   
                                                                                                                     2045             Y-JUN                     183166   
...                                                                                                                                                                ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                    1166825   
                                                                                                                     2040             Y-JUN                    1229571   
                                                                                                                     2045             Y-JUN                    1285676   
                                                                                         

### Projected population by 5-year age cohort

Load and validate the data.

In [94]:
(popCohort, popCohortResource, popCohortSchema) = morpc.frictionless.load_data(POP_COHORT_RESOURCE_PATH, archiveDir=inputDir, validate=True)

2026-08-25 22:37:27,737 | INFO | morpc.frictionless.frictionless.load_data: Loading Frictionless Resource file at location ..\morpc-projections-countypop\output_data\morpc-projections-countypop-5year.resource.yaml
2026-08-25 22:37:27,760 | INFO | morpc.frictionless.frictionless.load_data: Using schema path specified in resource file.
2026-08-25 22:37:27,763 | INFO | morpc.frictionless.frictionless.load_data: Copying data, resource file, and schema (if applicable) to directory input_data
2026-08-25 22:37:27,772 | INFO | morpc.frictionless.frictionless.load_data: --> Data file: input_data\morpc-projections-countypop-5year.csv
2026-08-25 22:37:27,774 | INFO | morpc.frictionless.frictionless.load_data: --> Resource file: input_data\morpc-projections-countypop-5year.resource.yaml
2026-08-25 22:37:27,776 | INFO | morpc.frictionless.frictionless.load_data: --> Schema file: input_data\morpc-projections-countypop-5year.schema.yaml
2026-08-25 22:37:27,778 | INFO | morpc.frictionless.frictionless

Inspect the raw data.

In [95]:
popCohort

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,AGEGROUP,POP,POPHH,POPGQ
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,0004,12877,12874,3
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,0509,16352,16352,0
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1014,17577,17574,3
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1519,16875,16578,297
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,2024,14204,13246,958
...,...,...,...,...,...,...,...,...,...,...
8689,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,6569,110214,109292,922
8690,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,7074,89319,88161,1158
8691,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,7579,68608,66741,1867
8692,2026,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2055,8084,54676,52094,2582


Melt the data to long form.

In [96]:
popCohort = popCohort.melt(id_vars=['VINTAGE', 'SCENARIO', 'GEOIDFQ', 'SUMLEVEL', 'NAME', 'YEAR', 'AGEGROUP'])
popCohort.head()

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,AGEGROUP,variable,value
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,0004,POP,12877
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,0509,POP,16352
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1014,POP,17577
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,1519,POP,16875
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,2024,POP,14204


Create composite variable names in standard form from the original variable names and the age group, then drop the original variable names and age group.

In [97]:
popCohort["standard_var"] = (popCohort["variable"].str.replace("POP","POPA") + "_" + popCohort["AGEGROUP"]).str.lower()
popCohort = popCohort.drop(columns=["AGEGROUP","variable"])
dataVars = list(popCohort["standard_var"].unique())

Pivot the table to wide form using the new variable names.

In [98]:
popCohort = popCohort.pivot(index=['VINTAGE', 'SCENARIO', 'GEOIDFQ', 'SUMLEVEL', 'NAME', 'YEAR'], columns="standard_var", values="value").reset_index()
popCohort.columns.name = None
popCohort.head()

,VINTAGE,SCENARIO,GEOIDFQ,SUMLEVEL,NAME,YEAR,popa_0004,popa_0509,popa_1014,popa_1519,...,popahh_4044,popahh_4549,popahh_5054,popahh_5559,popahh_6064,popahh_6569,popahh_7074,popahh_7579,popahh_8084,popahh_85o
0,2026,25th percentile migration,0500000US39041,050,Delaware County,2025,12877,16352,17577,16875,...,19469,17904,17703,14991,13816,12236,10319,7876,4198,3133
1,2026,25th percentile migration,0500000US39041,050,Delaware County,2030,11628,13789,18990,19961,...,20946,21784,18794,17658,14421,13366,11721,9674,6761,4441
2,2026,25th percentile migration,0500000US39041,050,Delaware County,2035,10003,12550,16437,21429,...,19292,23255,22641,18734,17030,13948,12792,10943,8227,6299
3,2026,25th percentile migration,0500000US39041,050,Delaware County,2040,10227,10936,15198,18901,...,15319,21612,24099,22528,18083,16473,13347,11926,9283,7560
4,2026,25th percentile migration,0500000US39041,050,Delaware County,2045,12152,11155,13584,17618,...,15685,17661,22473,23968,21795,17500,15731,12429,10107,8463


Adjust the identifier variables to match the common schema.

In [99]:
popCohort = popCohort.rename(columns={
    "VINTAGE":"vintage_period",
    "SCENARIO":"scenario",
    "GEOIDFQ":"geoidfq",
    "SUMLEVEL":"sumlevel",
    "NAME":"name",
    "YEAR":"reference_period"
})

popCohort["vintage_period_freq"] = VINTAGE_FREQ
popCohort["reference_period_freq"] = "Y-JUN"

Put the columns in an intuitive order.

In [100]:
popCohort = popCohort.filter(items=COMMON_ID_VARS + dataVars, axis="columns")

Use the common list of ID variables as the index.

In [101]:
popCohort = popCohort.set_index(COMMON_ID_VARS)

Inspect the transformed data.

In [102]:
popCohort

popa_0004  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq              
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                      12877   
                                                                                                                     2030             Y-JUN                      11628   
                                                                                                                     2035             Y-JUN                      10003   
                                                                                                                     2040             Y-JUN                      10227   
                                                                                                                     2045             Y-JUN                      12152   
...                                                                                                                                                                ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                     103281   
                                                                                                                     2040             Y-JUN                     104373   
                                                                                                                     2045             Y-JUN                     105179   
                                                                                                                     2050             Y-JUN                     106932   
                                                                                                                     2055             Y-JUN                     108158   

                                                                                                                                                             popa_0509  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq              
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                      16352   
                                                                                                                     2030             Y-JUN                      13789   
                                                                                                                     2035             Y-JUN                      12550   
                                                                                                                     2040             Y-JUN                      10936   
                                                                                                                     2045             Y-JUN                      11155   
...                                                                                                                                                                ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                      96211   
                                                                                                                     2040             Y-JUN                     100549   
                                                                                                                     2045             Y-JUN                     101639   
                                                                            

### Projected total jobs

Load and validate the data.

In [555]:
(jobsTotal, jobsTotalResource, jobsTotalGroupSchema) = morpc.frictionless.load_data(JOBS_TOTAL_RESOURCE_PATH, archiveDir=inputDir, validate=True)

2026-08-27 22:00:31,111 | INFO | morpc.frictionless.frictionless.load_data: Loading Frictionless Resource file at location ..\morpc-projections-countyjobs\output_data\morpc-projections-countyjobs.resource.yaml
2026-08-27 22:00:31,122 | INFO | morpc.frictionless.frictionless.load_data: Using schema path specified in resource file.
2026-08-27 22:00:31,123 | INFO | morpc.frictionless.frictionless.load_data: Copying data, resource file, and schema (if applicable) to directory input_data
2026-08-27 22:00:31,128 | INFO | morpc.frictionless.frictionless.load_data: --> Data file: input_data\morpc-projections-countyjobs.csv
2026-08-27 22:00:31,129 | INFO | morpc.frictionless.frictionless.load_data: --> Resource file: input_data\morpc-projections-countyjobs.resource.yaml
2026-08-27 22:00:31,130 | INFO | morpc.frictionless.frictionless.load_data: --> Schema file: input_data\morpc-projections-countyjobs.schema.yaml
2026-08-27 22:00:31,130 | INFO | morpc.frictionless.frictionless.load_data: Validat

Inspect the raw data.

In [556]:
jobsTotal

,vintage_period,vintage_period_freq,scenario,geoidfq,sumlevel,name,reference_period,reference_period_freq,jobs
0,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2025,Y-JUN,120259
1,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2030,Y-JUN,131966
2,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2035,Y-JUN,143270
3,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2040,Y-JUN,151892
4,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2045,Y-JUN,158346
...,...,...,...,...,...,...,...,...,...
478,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2035,Y-JUN,1229529
479,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2040,Y-JUN,1295176
480,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2045,Y-JUN,1351680
481,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2050,Y-JUN,1405071


Put the columns in an intuitive order.

In [557]:
dataVars = ["jobs"]
jobsTotal = jobsTotal.filter(items=COMMON_ID_VARS + dataVars, axis="columns")

Use the common list of ID variables as the index.

In [558]:
jobsTotal = jobsTotal.set_index(COMMON_ID_VARS)

Inspect the transformed data.

In [559]:
jobsTotal

jobs
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq         
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   120259
                                                                                                                     2030             Y-JUN                   131966
                                                                                                                     2035             Y-JUN                   143270
                                                                                                                     2040             Y-JUN                   151892
                                                                                                                     2045             Y-JUN                   158346
...                                                                                                                                                              ...
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1229529
                                                                                                                     2040             Y-JUN                  1295176
                                                                                                                     2045             Y-JUN                  1351680
                                                                                                                     2050             Y-JUN                  1405071
                                                                                                                     2055             Y-JUN                  1455139

[483 rows x 1 columns]

### Census ACS 5-year estimates

#### Average household size

Retrieve the data from the Census API.

In [275]:
acsTable = "B25010"
ep  = morpc_census.Endpoint(f'acs/{ACS_SURVEY}', ACS_YEAR)
grp = morpc_census.Group(ep, acsTable)
api = morpc_census.CensusAPI(ep, morpc_census.SCOPES['region15'], group=grp, sumlevel=morpc_census.SumLevel('county'))

2026-08-26 09:31:32,129 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.__init__: Initializing CensusAPI for census-acs-acs5-2024-county-region15-b25010.
2026-08-26 09:31:32,129 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.__init__: Building request URL and parameters.
2026-08-26 09:31:32,537 | INFO | morpc_census.geos.geoinfo_from_scope_sumlevel: SumLevel 'county' specified for scope 'region15'.
2026-08-26 09:31:32,539 | WARNING | morpc_census.geos.geoinfo_from_scope_sumlevel: Scope 'region15' is already at 'county'; using scope parameters directly.
2026-08-26 09:31:32,541 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.__init__: Fetching data from https://api.census.gov/data/2024/acs/acs5? with params {'get': 'group(B25010)', 'for': 'county:041, 045, 047, 049, 073, 083, 089, 091, 097, 101, 117, 127, 129, 141, 159', 'in': 'state:39'}.
2026-08-26 09:31:32,544 | INFO | morpc_census.api.Cen

Create an archival copy of the raw data.

In [276]:
api.save(inputDir)

2026-08-26 09:31:33,325 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.save: Writing data to input_data\census-acs-acs5-2024-county-region15-b25010.long.csv.
2026-08-26 09:31:33,333 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.save: Writing schema to input_data\census-acs-acs5-2024-county-region15-b25010.schema.yaml.
2026-08-26 09:31:33,335 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.define_schema: Defining schema for acs/acs5 / 2024 / B25010.
2026-08-26 09:31:33,365 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.save: Writing resource to input_data\census-acs-acs5-2024-county-region15-b25010.resource.yaml.
2026-08-26 09:31:33,413 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25010.save: Save complete and resource validated.


Create a working dataframe of the data.

In [277]:
temp = api.long.copy()

Inspect the data.

In [278]:
temp.head()

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.73,0.02
1,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:!!Owner occupied,B25010_002,2.90,0.03
2,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:!!Renter occu...,B25010_003,2.12,0.09
3,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.65,0.02
4,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:!!Owner occupied,B25010_002,2.78,0.04


Create a dimension table from the long-form data.

In [279]:
temp = morpc_census.DimensionTable(temp)

Inspect the dimensions for each variable.

In [280]:
temp.dims

,Average household size,Total,Tenure
variable,,,
B25010_001,Average household size --,Total,
B25010_002,Average household size --,Total,Owner occupied
B25010_003,Average household size --,Total,Renter occupied


Extract only the totals.

In [281]:
temp = temp.long
temp = temp.loc[temp["variable"] == "B25010_001"].copy()
temp.head()

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.73,0.02
3,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.65,0.02
6,0500000US39047,"Fayette County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.46,0.06
9,0500000US39049,"Franklin County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.36,0.01
12,0500000US39073,"Hocking County, Ohio",2024,acs/acs5,Average household size of occupied housing uni...,Occupied housing units,Average household size --!!Total:,B25010_001,2.38,0.05


Delete some fields that we don't need. 

In [282]:
temp = temp.filter(items=["geoidfq","estimate"], axis="columns")

Give the estimate field a unique name.

In [283]:
temp = temp.rename(columns={"estimate":"hhsize"})

Store the data in a persistent dataframe and delete the temporary dataframe.

In [284]:
acsHouseholdSize = temp.copy()
del(temp)

Use the fully-qualified GEOID as the index.

In [285]:
acsHouseholdSize = acsHouseholdSize.set_index("geoidfq")

Inspect the final data.

In [286]:
acsHouseholdSize.head()

,hhsize
geoidfq,
0500000US39041,2.73
0500000US39045,2.65
0500000US39047,2.46
0500000US39049,2.36
0500000US39073,2.38


#### Housing unit occupancy rate

Download the ACS table and create a dimension table as described above.

In [287]:
acsTable = "B25002"

# Retrieve the data from the API
ep  = morpc_census.Endpoint(f'acs/{ACS_SURVEY}', ACS_YEAR)
grp = morpc_census.Group(ep, acsTable)
api = morpc_census.CensusAPI(ep, morpc_census.SCOPES['region15'], group=grp, sumlevel=morpc_census.SumLevel('county'))

# Create an archival copy of the raw data.
api.save(inputDir)

# Create a working dataframe of the data.
temp = api.long.copy()

# Create a dimension table from the long-form data.
temp = morpc_census.DimensionTable(temp)

2026-08-26 09:31:36,805 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25002.__init__: Initializing CensusAPI for census-acs-acs5-2024-county-region15-b25002.
2026-08-26 09:31:36,806 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25002.__init__: Building request URL and parameters.
2026-08-26 09:31:37,204 | INFO | morpc_census.geos.geoinfo_from_scope_sumlevel: SumLevel 'county' specified for scope 'region15'.
2026-08-26 09:31:37,204 | WARNING | morpc_census.geos.geoinfo_from_scope_sumlevel: Scope 'region15' is already at 'county'; using scope parameters directly.
2026-08-26 09:31:37,204 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b25002.__init__: Fetching data from https://api.census.gov/data/2024/acs/acs5? with params {'get': 'group(B25002)', 'for': 'county:041, 045, 047, 049, 073, 083, 089, 091, 097, 101, 117, 127, 129, 141, 159', 'in': 'state:39'}.
2026-08-26 09:31:37,204 | INFO | morpc_census.api.Cen

Inspect the dimensions for each variable.

In [288]:
temp.dims

,Total,Occupancy Status
variable,,
B25002_001,Total,
B25002_002,Total,Occupied
B25002_003,Total,Vacant


Extract the total number of housing units and the number of occupied housing units.

In [289]:
temp = temp.long
temp = temp.loc[temp["variable"].isin(["B25002_001","B25002_002"])].copy()
temp.head()

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Occupancy status,Housing units,Total:,B25002_001,86269.0,93.0
1,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Occupancy status,Housing units,Total:!!Occupied,B25002_002,82287.0,534.0
3,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Occupancy status,Housing units,Total:,B25002_001,64036.0,99.0
4,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Occupancy status,Housing units,Total:!!Occupied,B25002_002,60357.0,513.0
6,0500000US39047,"Fayette County, Ohio",2024,acs/acs5,Occupancy status,Housing units,Total:,B25002_001,12725.0,50.0


Delete some fields that we don't need. 

In [290]:
temp = temp.filter(items=["geoidfq","variable","estimate"], axis="columns")

Pivot the table to wide form such that total housing units and occupied housing units are in their own columns.

In [291]:
temp = temp.pivot(index="geoidfq", columns="variable", values="estimate")
temp.columns.name = None
temp = temp.rename(columns={
    "B25002_001":"hu",
    "B25002_002":"huocc"
})

Compute the housing unit occupancy rate by dividing the number of occupied units by the total number of units.

In [292]:
temp["huocc_rate"] = temp["huocc"] / temp["hu"]

Inspect the data.

In [293]:
temp.head()

,hu,huocc,huocc_rate
geoidfq,,,
0500000US39041,86269.0,82287.0,0.953842
0500000US39045,64036.0,60357.0,0.942548
0500000US39047,12725.0,11582.0,0.910177
0500000US39049,595469.0,553099.0,0.928846
0500000US39073,13013.0,11568.0,0.888957


In [294]:
temp.describe()

,hu,huocc,huocc_rate
count,15.000000,15.000000,15.000000
mean,69905.666667,64919.133333,0.919184
std,147219.201330,136852.131515,0.036265
min,12725.000000,11568.000000,0.819576
25%,15781.500000,14574.500000,0.901510
50%,25158.000000,23491.000000,0.928830
75%,48064.000000,45082.500000,0.946061
max,595469.000000,553099.000000,0.963193


Extract only the occupancy rate. We no longer require the other variables.

In [295]:
temp = temp.filter(items=["geoidfq","huocc_rate"], axis="columns")

Store the data in a persistent dataframe and delete the temporary dataframe.

In [296]:
acsOccupancyRate = temp.copy()
del(temp)

Inspect the final data.

In [297]:
acsOccupancyRate.head()

,huocc_rate
geoidfq,
0500000US39041,0.953842
0500000US39045,0.942548
0500000US39047,0.910177
0500000US39049,0.928846
0500000US39073,0.888957


#### Labor force participation rate and employment rate

Download the ACS table and create a dimension table as described above.

In [343]:
acsTable = "B23025"

# Retrieve the data from the API
ep  = morpc_census.Endpoint(f'acs/{ACS_SURVEY}', ACS_YEAR)
grp = morpc_census.Group(ep, acsTable)
api = morpc_census.CensusAPI(ep, morpc_census.SCOPES['region15'], group=grp, sumlevel=morpc_census.SumLevel('county'))

# Create an archival copy of the raw data.
api.save(inputDir)

# Create a working dataframe of the data.
temp = api.long.copy()

# Create a dimension table from the long-form data.
temp = morpc_census.DimensionTable(temp)

2026-08-26 09:43:14,582 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b23025.__init__: Initializing CensusAPI for census-acs-acs5-2024-county-region15-b23025.
2026-08-26 09:43:14,583 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b23025.__init__: Building request URL and parameters.
2026-08-26 09:43:14,965 | INFO | morpc_census.geos.geoinfo_from_scope_sumlevel: SumLevel 'county' specified for scope 'region15'.
2026-08-26 09:43:14,967 | WARNING | morpc_census.geos.geoinfo_from_scope_sumlevel: Scope 'region15' is already at 'county'; using scope parameters directly.
2026-08-26 09:43:14,967 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2024-county-region15-b23025.__init__: Fetching data from https://api.census.gov/data/2024/acs/acs5? with params {'get': 'group(B23025)', 'for': 'county:041, 045, 047, 049, 073, 083, 089, 091, 097, 101, 117, 127, 129, 141, 159', 'in': 'state:39'}.
2026-08-26 09:43:14,969 | INFO | morpc_census.api.Cen

Inspect the dimensions for each variable.

In [344]:
temp.dims

,Total,Labor Force Participation,Labor Force Component,Employment Status
variable,,,,
B23025_001,Total,,,
B23025_002,Total,In labor force,,
B23025_003,Total,In labor force,Civilian labor force,
B23025_004,Total,In labor force,Civilian labor force,Employed
B23025_005,Total,In labor force,Civilian labor force,Unemployed
B23025_006,Total,In labor force,Armed Forces,
B23025_007,Total,Not in labor force,,


Extract the total number of people in the labor force and not in the labor force.

In [345]:
temp = temp.long
temp = temp.loc[temp["variable"].isin(["B23025_002","B23025_007","B23025_004","B23025_006"])].copy()
temp.head()

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
1,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Employment status for the population 16 years ...,Population 16 years and over,Total:!!In labor force:,B23025_002,124118.0,1464.0
4,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Employment status for the population 16 years ...,Population 16 years and over,Total:!!In labor force:!!Civilian labor force:...,B23025_004,120222.0,1448.0
2,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Employment status for the population 16 years ...,Population 16 years and over,Total:!!In labor force:!!Armed Forces,B23025_006,216.0,120.0
6,0500000US39041,"Delaware County, Ohio",2024,acs/acs5,Employment status for the population 16 years ...,Population 16 years and over,Total:!!Not in labor force,B23025_007,53355.0,1548.0
8,0500000US39045,"Fairfield County, Ohio",2024,acs/acs5,Employment status for the population 16 years ...,Population 16 years and over,Total:!!In labor force:,B23025_002,83076.0,1129.0


Delete some fields that we don't need. 

In [346]:
temp = temp.filter(items=["geoidfq","variable","estimate"], axis="columns")

Pivot the table to wide form such that labor force participants and non-participants are in their own columns.

In [347]:
temp = temp.pivot(index="geoidfq", columns="variable", values="estimate")
temp.columns.name = None
temp = temp.rename(columns={
    "B23025_002":"lbfrc",
    "B23025_007":"lbfrc_non",
    "B23025_004":"emp_civ",
    "B23025_006":"emp_mil"
})

Compute the labor force participation rate by dividing the number of labor force participants by the combined number of labor force participants and non-participants (i.e. the total population over the age of 16)

In [348]:
temp["lbfrc_rate"] = temp["lbfrc"] / (temp["lbfrc"] + temp["lbfrc_non"])

Compute the employment rate by dividing the total number of employed persons (civilian plus military) by the number of persons in the labor force.

In [349]:
temp["emp_rate"] = (temp["emp_civ"] + temp["emp_mil"]) / temp["lbfrc"]

Extract only the rates. We no longer require the other variables.

In [350]:
temp = temp.filter(items=["geoidfq","lbfrc_rate","emp_rate"], axis="columns")

Inspect the data.

In [351]:
temp.head()

,lbfrc_rate,emp_rate
geoidfq,,
0500000US39041,0.699363,0.970351
0500000US39045,0.641117,0.961830
0500000US39047,0.584962,0.962798
0500000US39049,0.694550,0.954489
0500000US39073,0.575662,0.945816


In [352]:
temp.describe()

,lbfrc_rate,emp_rate
count,15.000000,15.000000
mean,0.609282,0.962259
std,0.052143,0.010299
min,0.531982,0.936766
25%,0.580312,0.960549
50%,0.600012,0.962997
75%,0.636226,0.969022
max,0.699363,0.975245


Store the data in a persistent dataframe and delete the temporary dataframe.

In [353]:
acsEmployment = temp.copy()
del(temp)

Inspect the final data.

In [354]:
acsEmployment.head()

,lbfrc_rate,emp_rate
geoidfq,,
0500000US39041,0.699363,0.970351
0500000US39045,0.641117,0.961830
0500000US39047,0.584962,0.962798
0500000US39049,0.694550,0.954489
0500000US39073,0.575662,0.945816


#### Combine the ACS variables

Put both ACS variables in a single table indexed by fully-qualified GEOID.

In [355]:
acs = acsHouseholdSize.join(acsOccupancyRate).join(acsEmployment)
acs.head()

,hhsize,huocc_rate,lbfrc_rate,emp_rate
geoidfq,,,,
0500000US39041,2.73,0.953842,0.699363,0.970351
0500000US39045,2.65,0.942548,0.641117,0.961830
0500000US39047,2.46,0.910177,0.584962,0.962798
0500000US39049,2.36,0.928846,0.694550,0.954489
0500000US39073,2.38,0.888957,0.575662,0.945816


### Mid-year population estimates (DELETE)

Load and validate the data.

Inspect the raw data.

Extract only the population estimates.

Extract only the MORPC counties.

Extract the most recent vintage available for each year.

Rename the value variable to indicate that it contains population values.

Extract only the columns that we require.

Use the fully qualified GEOID and year as the index.

Inspect the transformed data.

### Standardized BLS LAUS data (DELETE)

Load and validate the data.

Inspect the raw data.

Extract only the county level data, then drop the SUMLEVEL column.

Extract the most recent 5 years from the data. If fewer than 5 years are available, use all available years.

Join the mid-year population estimates (needed to compute labor force participation rate).

Verify that a population value is available for all years.

Compute labor force participation rate.

Compute the employment rate.

Extract only the rates.

In [266]:
laus

lbfrc_rate  emp_rate
GEOIDFQ        YEAR                      
0500000US39041 2019    0.538333  0.968280
               2020    0.519832  0.946477
               2021    0.528061  0.962774
               2022    0.530473  0.969913
               2023    0.534452  0.970823
...                         ...       ...
0500000US39159 2020    0.518258  0.946950
               2021    0.523924  0.966401
               2022    0.527442  0.971961
               2023    0.531262  0.973287
               2024    0.524568  0.966711

[90 rows x 2 columns]

## Transform data

### Derive projections of households and housing units

Start with total household population.

In [422]:
hh = popTotal.copy().filter(items=["pophh"], axis="columns")
hh

pophh
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq         
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   239707
                                                                                                                     2030             Y-JUN                   256602
                                                                                                                     2035             Y-JUN                   268942
                                                                                                                     2040             Y-JUN                   278970
                                                                                                                     2045             Y-JUN                   289205
...                                                                                                                                                              ...
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1766110
                                                                                                                     2040             Y-JUN                  1847669
                                                                                                                     2045             Y-JUN                  1921714
                                                                                                                     2050             Y-JUN                  1990641
                                                                                                                     2055             Y-JUN                  2054441

[483 rows x 1 columns]

The dataset includes projections for various county-based regions.  Extract only data for individual counties.

In [423]:
hh = hh.xs(morpc.SUMLEVEL_LOOKUP["COUNTY"], level="sumlevel", drop_level=False)

Merge the ACS household size estimates and housing unit occupancy rates.

In [424]:
hh = hh.merge(acs.filter(items=["hhsize","huocc_rate"], axis="columns"), left_index=True, right_index=True, how="left")

Verify that a household sizes and occupancy rates are defined for each county.

In [425]:
errorFlag = False
for var in ["hhsize","huocc_rate"]:
    if(not hh.loc[hh[var].isna()].empty):
        errorFlag = True
        logger.error(f"Required variable {var} is missing for one or more counties.")
if(errorFlag == True):
    raise RuntimeError
else:
    logger.info("Required variables are available for all counties.") 

2026-08-26 10:40:48,671 | INFO | __main__.<module>: Required variables are available for all counties.


Estimate the number of households by dividing the household population by the average household size.

In [426]:
hh["hh"] = (hh["pophh"] / hh["hhsize"]).round(0).astype("int")

Estimate the number of housing units by dividing the number of households by the housing unit occupancy rate.  Note that this implicitly assumes one household per occupied housing unit.

In [427]:
hh["hu"] = (hh["hh"] / hh["huocc_rate"]).round(0).astype("int")

Extract only the estimates of households and housing units, dropping the other variables, which are no longer required.

In [428]:
hh = hh.filter(items=["hh","hu"], axis="columns")

Inspect the data.

In [429]:
hh

hh  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name            reference_period reference_period_freq           
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County 2025             Y-JUN                   87805   
                                                                                                     2030             Y-JUN                   93993   
                                                                                                     2035             Y-JUN                   98514   
                                                                                                     2040             Y-JUN                  102187   
                                                                                                     2045             Y-JUN                  105936   
...                                                                                                                                             ...   
                                   75th percentile migration 0500000US39159 050      Union County    2035             Y-JUN                   32372   
                                                                                                     2040             Y-JUN                   34991   
                                                                                                     2045             Y-JUN                   37548   
                                                                                                     2050             Y-JUN                   40183   
                                                                                                     2055             Y-JUN                   42884   

                                                                                                                                                 hu  
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name            reference_period reference_period_freq          
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County 2025             Y-JUN                   92054  
                                                                                                     2030             Y-JUN                   98541  
                                                                                                     2035             Y-JUN                  103281  
                                                                                                     2040             Y-JUN                  107132  
                                                                                                     2045             Y-JUN                  111062  
...                                                                                                                                             ...  
                                   75th percentile migration 0500000US39159 050      Union County    2035             Y-JUN                   33609  
                                                                                                     2040             Y-JUN                   36328  
                                                                                                     2045             Y-JUN                   38983  
                                                                                                     2050             Y-JUN                   41719  
                                                                                                     2055             Y-JUN                   44523  

[315 rows x 2 columns]

Show the descriptive statistics for each variable for each scenario.

In [430]:
for var in ["hh","hu"]:
    print(("Households" if var=="hh" else "Housing units"))
    firstTime = True
    for scenario in hh.index.get_level_values("scenario").unique():
        scenarioStats = hh.xs(scenario, level="scenario", drop_level=False).filter(items=[var], axis="columns").describe().rename(columns={var:scenario})
        if(firstTime == True):
            allStats = scenarioStats.copy()
            firstTime = False
        else:
            allStats = allStats.join(scenarioStats)
    display(allStats)

Households


,25th percentile migration,50th percentile migration,75th percentile migration
count,105.000000,105.000000,105.000000
mean,69705.428571,72230.028571,75083.228571
std,147824.742992,152441.142158,157550.104671
min,7824.000000,8739.000000,9494.000000
25%,13355.000000,13790.000000,14201.000000
50%,22393.000000,23106.000000,24474.000000
75%,64706.000000,65238.000000,66270.000000
max,640936.000000,680085.000000,721842.000000


Housing units


,25th percentile migration,50th percentile migration,75th percentile migration
count,105.000000,105.000000,105.000000
mean,74959.047619,77682.285714,80735.171429
std,159026.809119,163993.566863,169486.567027
min,8801.000000,9831.000000,10680.000000
25%,14586.000000,15196.000000,15510.000000
50%,23848.000000,24680.000000,26816.000000
75%,68650.000000,69215.000000,70309.000000
max,690035.000000,732183.000000,777139.000000


### Derive projections of labor force and workers

Start with population by 5-year cohort.

In [450]:
emp = popCohort.copy()
emp

popa_0004  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq              
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                      12877   
                                                                                                                     2030             Y-JUN                      11628   
                                                                                                                     2035             Y-JUN                      10003   
                                                                                                                     2040             Y-JUN                      10227   
                                                                                                                     2045             Y-JUN                      12152   
...                                                                                                                                                                ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                     103281   
                                                                                                                     2040             Y-JUN                     104373   
                                                                                                                     2045             Y-JUN                     105179   
                                                                                                                     2050             Y-JUN                     106932   
                                                                                                                     2055             Y-JUN                     108158   

                                                                                                                                                             popa_0509  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq              
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                      16352   
                                                                                                                     2030             Y-JUN                      13789   
                                                                                                                     2035             Y-JUN                      12550   
                                                                                                                     2040             Y-JUN                      10936   
                                                                                                                     2045             Y-JUN                      11155   
...                                                                                                                                                                ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                      96211   
                                                                                                                     2040             Y-JUN                     100549   
                                                                                                                     2045             Y-JUN                     101639   
                                                                            

The dataset includes projections for various county-based regions.  Extract only data for individual counties.

In [451]:
emp = emp.xs(morpc.SUMLEVEL_LOOKUP["COUNTY"], level="sumlevel", drop_level=False)

The dataset includes group quarters population by age and household population by age in addition to total population by age.  Extract only total population by age.

In [452]:
emp = emp.filter(regex="popa_", axis="columns")

For employment, the universe for analysis is the population 16 years and over. Drop the age groups which are not included in that population.

In [453]:
emp = emp.drop(columns=["popa_0004","popa_0509","popa_1014"])

The youngest age group (15 to 19) includes some population under the age of 16.  Create a new group which approximates the population 16 and older by scaling the 15 to 19 group by 4/5. Then delete the 15 to 19 group.

In [454]:
emp["popa_1619"] = (emp["popa_1519"] * 4/5).round(0).astype("int")
emp = emp.drop(columns="popa_1519")

Create a new group (16 and over) which represents the universe of the working age population by summing the remaining, then extract only the new column.

In [456]:
emp["popa_16o"] = emp.sum(axis="columns")
emp = emp.filter(items=["popa_16o"], axis="columns")

Merge the ACS labor force participation rates and employment rates.

In [458]:
emp = emp.merge(acs.filter(items=["lbfrc_rate","emp_rate"], axis="columns"), left_index=True, right_index=True, how="left")

Verify that labor force paticipation rates and employment rates are defined for each county.

In [459]:
errorFlag = False
for var in ["lbfrc_rate","emp_rate"]:
    if(not emp.loc[emp[var].isna()].empty):
        errorFlag = True
        logger.error(f"Required variable {var} is missing for one or more counties.")
if(errorFlag == True):
    raise RuntimeError
else:
    logger.info("Required variables are available for all counties.") 

2026-08-26 10:59:17,770 | INFO | __main__.<module>: Required variables are available for all counties.


Estimate the number of labor force participants by multiplying the working age population by the labor force participation rate.

In [460]:
emp["lbfrc"] = (emp["popa_16o"] * emp["lbfrc_rate"]).round(0).astype("int")

Estimate the number of workers (employed persons) by multiplying the number of labor force participants by the employment rate.

In [461]:
emp["wrkrs"] = (emp["lbfrc"] * emp["emp_rate"]).round(0).astype("int")

Extract only the estimates of labor force and employed workers, dropping the other variables, which are no longer required.

In [462]:
emp = emp.filter(items=["lbfrc","wrkrs"], axis="columns")

Inspect the data.

In [463]:
emp

lbfrc  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name            reference_period reference_period_freq           
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County 2025             Y-JUN                  134173   
                                                                                                     2030             Y-JUN                  147235   
                                                                                                     2035             Y-JUN                  159847   
                                                                                                     2040             Y-JUN                  169467   
                                                                                                     2045             Y-JUN                  176667   
...                                                                                                                                             ...   
                                   75th percentile migration 0500000US39159 050      Union County    2035             Y-JUN                   50459   
                                                                                                     2040             Y-JUN                   55522   
                                                                                                     2045             Y-JUN                   60023   
                                                                                                     2050             Y-JUN                   64053   
                                                                                                     2055             Y-JUN                   67834   

                                                                                                                                              wrkrs  
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name            reference_period reference_period_freq          
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County 2025             Y-JUN                  130195  
                                                                                                     2030             Y-JUN                  142870  
                                                                                                     2035             Y-JUN                  155108  
                                                                                                     2040             Y-JUN                  164442  
                                                                                                     2045             Y-JUN                  171429  
...                                                                                                                                             ...  
                                   75th percentile migration 0500000US39159 050      Union County    2035             Y-JUN                   49156  
                                                                                                     2040             Y-JUN                   54088  
                                                                                                     2045             Y-JUN                   58473  
                                                                                                     2050             Y-JUN                   62399  
                                                                                                     2055             Y-JUN                   66082  

[315 rows x 2 columns]

Show the descriptive statistics for each variable for each scenario.

In [464]:
for var in ["lbfrc","wrkrs"]:
    print(("Labor force" if var=="lbfrc" else "Workers"))
    firstTime = True
    for scenario in emp.index.get_level_values("scenario").unique():
        scenarioStats = emp.xs(scenario, level="scenario", drop_level=False).filter(items=[var], axis="columns").describe().rename(columns={var:scenario})
        if(firstTime == True):
            allStats = scenarioStats.copy()
            firstTime = False
        else:
            allStats = allStats.join(scenarioStats)
    display(allStats)

Labor force


,25th percentile migration,50th percentile migration,75th percentile migration
count,105.000000,105.000000,1.050000e+02
mean,96923.447619,100337.285714,1.042923e+05
std,204633.183848,210945.572250,2.179870e+05
min,8914.000000,9974.000000,1.084800e+04
25%,17500.000000,18214.000000,1.916600e+04
50%,28142.000000,30016.000000,3.078400e+04
75%,90641.000000,91373.000000,9.279100e+04
max,902534.000000,955435.000000,1.011854e+06


Workers


,25th percentile migration,50th percentile migration,75th percentile migration
count,105.000000,105.000000,105.000000
mean,92973.085714,96250.638095,100051.628571
std,195332.834025,201354.154018,208079.095520
min,8431.000000,9434.000000,10260.000000
25%,16991.000000,17684.000000,18609.000000
50%,27120.000000,28993.000000,29766.000000
75%,87181.000000,87885.000000,89249.000000
max,861458.000000,911952.000000,965803.000000


### Combine all variables in a single semi-long table

Join the various datasets on their common index.

In [560]:
countyControls = popTotal.join(popCohort).join(popLargeGroup).join(hh).join(emp).join(jobsTotal)

Inspect the combined data.

In [561]:
countyControls

pop  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   242032   
                                                                                                                     2030             Y-JUN                   258927   
                                                                                                                     2035             Y-JUN                   271837   
                                                                                                                     2040             Y-JUN                   282457   
                                                                                                                     2045             Y-JUN                   293027   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1801228   
                                                                                                                     2040             Y-JUN                  1885583   
                                                                                                                     2045             Y-JUN                  1961073   
                                                                                                                     2050             Y-JUN                  2030658   
                                                                                                                     2055             Y-JUN                  2095623   

                                                                                                                                                               pophh  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   239707   
                                                                                                                     2030             Y-JUN                   256602   
                                                                                                                     2035             Y-JUN                   268942   
                                                                                                                     2040             Y-JUN                   278970   
                                                                                                                     2045             Y-JUN                   289205   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1766110   
                                                                                                                     2040             Y-JUN                  1847669   
                                                                                                                     2045             Y-JUN                  1921714   
                                                                                                                     2050       

Review the list of data columns.

In [502]:
countyControls.columns

Index(['pop', 'pophh', 'popgq', 'popa_0004', 'popa_0509', 'popa_1014',
       'popa_1519', 'popa_2024', 'popa_2529', 'popa_3034', 'popa_3539',
       'popa_4044', 'popa_4549', 'popa_5054', 'popa_5559', 'popa_6064',
       'popa_6569', 'popa_7074', 'popa_7579', 'popa_8084', 'popa_85o',
       'popahh_0004', 'popahh_0509', 'popahh_1014', 'popahh_1519',
       'popahh_2024', 'popahh_2529', 'popahh_3034', 'popahh_3539',
       'popahh_4044', 'popahh_4549', 'popahh_5054', 'popahh_5559',
       'popahh_6064', 'popahh_6569', 'popahh_7074', 'popahh_7579',
       'popahh_8084', 'popahh_85o', 'popagq_0004', 'popagq_0509',
       'popagq_1014', 'popagq_1519', 'popagq_2024', 'popagq_2529',
       'popagq_3034', 'popagq_3539', 'popagq_4044', 'popagq_4549',
       'popagq_5054', 'popagq_5559', 'popagq_6064', 'popagq_6569',
       'popagq_7074', 'popagq_7579', 'popagq_8084', 'popagq_85o', 'popa_17u',
       'popa_1864', 'popa_65o', 'popahh_17u', 'popahh_1864', 'popahh_65o',
       'popagq_17u', 'popa

Review the list of unique values for select identifier columns.

In [503]:
for var in ["vintage_period", "scenario", "name", "reference_period"]:
    print(f"{var}: {list(countyControls.index.get_level_values(var).unique())}\n")

vintage_period: [2026]

scenario: ['25th percentile migration', '50th percentile migration', '75th percentile migration']

name: ['Delaware County', 'Fairfield County', 'Fayette County', 'Franklin County', 'Hocking County', 'Knox County', 'Licking County', 'Logan County', 'Madison County', 'Marion County', 'Morrow County', 'Perry County', 'Pickaway County', 'Ross County', 'Union County', 'Core-based statistical area', 'MORPC 15-county region', 'MORPC 10-county region', 'MORPC 7-county region', 'MORPC CORPO region', 'MORPC CEDS region', 'OneColumbus region', 'Franklin/Delaware county region']

reference_period: [2025, 2030, 2035, 2040, 2045, 2050, 2055]



### Summarize projections for county-based regions

Some of the variables include projections for county-based regions, however these have not yet been produced for the derived variables.  Compute missing projections for each of the county-based regions by summarizing the county-level projections.

First, identify the variables that have null values.

In [562]:
temp = countyControls.copy()
varList = []
for var in countyControls.columns:
    if(not temp.loc[temp[var].isna()].empty):
        varList.append(var)
varList

['hh', 'hu', 'lbfrc', 'wrkrs']

For the variables with null values, iterate through each of the county-based regions and summarize the county projections for the variables for that region.

In [563]:
temp = countyControls.filter(items=varList, axis="columns")

for region in morpc.CONST_REGIONS_COUNTYBASED:
    regionCounties = morpc.CONST_REGIONS[region]
    logger.info(f"Summarizing projections for region {region}")
    # Extract the counties in the region
    thisRegion = temp[temp.index.get_level_values('name').isin([f"{x} County" for x in regionCounties])].reset_index()
    # Summarize the projections for the region
    thisRegion = thisRegion.drop(columns=['geoidfq','sumlevel','name']).groupby(['vintage_period','vintage_period_freq','scenario','reference_period','reference_period_freq']).sum().reset_index()
    # Assign the sumlevel, name, and GEOID for the region
    sumlevel = morpc.SUMLEVEL_LOOKUP[region]
    thisRegion["sumlevel"] = sumlevel
    thisRegion["name"] = morpc.SUMLEVEL_DESCRIPTIONS[sumlevel]["singular"]
    thisRegion["geoidfq"] = sumlevel + "0000US" + morpc.CONST_REGIONS_GEOID[region]
    # Use the common index
    thisRegion = thisRegion.set_index(COMMON_ID_VARS)
    # Update the existing county controls table with the new values
    countyControls.update(thisRegion, overwrite=False, errors="raise")

for var in varList:
    countyControls[var] = countyControls[var].astype("int")

2026-08-27 22:00:56,694 | INFO | __main__.<module>: Summarizing projections for region REGION15
2026-08-27 22:00:56,752 | INFO | __main__.<module>: Summarizing projections for region REGION10
2026-08-27 22:00:56,780 | INFO | __main__.<module>: Summarizing projections for region REGION7
2026-08-27 22:00:56,801 | INFO | __main__.<module>: Summarizing projections for region REGIONCEDS
2026-08-27 22:00:56,832 | INFO | __main__.<module>: Summarizing projections for region REGIONCORPO
2026-08-27 22:00:56,851 | INFO | __main__.<module>: Summarizing projections for region REGIONONECBUS
2026-08-27 22:00:56,869 | INFO | __main__.<module>: Summarizing projections for region CBSA
2026-08-27 22:00:56,888 | INFO | __main__.<module>: Summarizing projections for region REGIONFRADEL


Inspect the data.

In [564]:
countyControls

pop  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   242032   
                                                                                                                     2030             Y-JUN                   258927   
                                                                                                                     2035             Y-JUN                   271837   
                                                                                                                     2040             Y-JUN                   282457   
                                                                                                                     2045             Y-JUN                   293027   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1801228   
                                                                                                                     2040             Y-JUN                  1885583   
                                                                                                                     2045             Y-JUN                  1961073   
                                                                                                                     2050             Y-JUN                  2030658   
                                                                                                                     2055             Y-JUN                  2095623   

                                                                                                                                                               pophh  \
vintage_period vintage_period_freq scenario                  geoidfq        sumlevel name                            reference_period reference_period_freq            
2026           Y-DEC               25th percentile migration 0500000US39041 050      Delaware County                 2025             Y-JUN                   239707   
                                                                                                                     2030             Y-JUN                   256602   
                                                                                                                     2035             Y-JUN                   268942   
                                                                                                                     2040             Y-JUN                   278970   
                                                                                                                     2045             Y-JUN                   289205   
...                                                                                                                                                              ...   
                                   75th percentile migration M090000US001   M09      Franklin/Delaware county region 2035             Y-JUN                  1766110   
                                                                                                                     2040             Y-JUN                  1847669   
                                                                                                                     2045             Y-JUN                  1921714   
                                                                                                                     2050       

## Export data

### Wide form table

Load the output schema.

In [573]:
outputWideSchema = morpc.frictionless.load_schema(OUTPUT_WIDE_SCHEMA_PATH)

Create a working copy of the county controls data and reset the index.

In [574]:
outputWide = countyControls.copy().reset_index()

Extract only the fields required by the schema and ensure they are in the correct order.

In [575]:
outputWide = outputWide.filter(items=outputWideSchema.field_names, axis="columns")

Ensure that all fields are cast as the data types required by the schema.

In [576]:
outputWide = morpc.frictionless.cast_field_types(outputWide, outputWideSchema)

2026-08-27 22:02:38,572 | INFO | morpc.frictionless.frictionless.cast_field_types: handleMissingValues set to True, converting [''] to np.nan


Sort the records in a logical order.

In [577]:
outputWide = outputWide.sort_values(by=outputWideSchema.primary_key)

Inspect the data.

In [578]:
outputWide

,vintage_period,vintage_period_freq,scenario,geoidfq,sumlevel,name,reference_period,reference_period_freq,pop,pophh,...,popahh_1864,popahh_65o,popagq_17u,popagq_1864,popagq_65o,hh,hu,lbfrc,wrkrs,jobs
0,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2025,Y-JUN,242032,239707,...,148514,37762,124,1342,859,87805,92054,134173,130195,120259
1,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2030,Y-JUN,258927,256602,...,158373,45963,124,1342,859,93993,98541,147235,142870,131966
2,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2035,Y-JUN,271837,268942,...,169319,52209,146,1613,1136,98514,103281,159847,155108,143270
3,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2040,Y-JUN,282457,278970,...,176616,58589,155,1878,1454,102187,107132,169467,164442,151892
4,2026,Y-DEC,25th percentile migration,0500000US39041,050,Delaware County,2045,Y-JUN,293027,289205,...,181175,64230,138,1991,1693,105936,111062,176667,171429,158346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2035,Y-JUN,1801228,1766110,...,1144258,281108,3728,22568,8822,732173,785350,1029359,985166,1229529
479,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2040,Y-JUN,1885583,1847669,...,1205915,299367,3791,23657,10466,765780,821360,1085667,1039128,1295176
480,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2045,Y-JUN,1961073,1921714,...,1261627,311595,3557,24049,11753,796178,853910,1133974,1085415,1351680
481,2026,Y-DEC,75th percentile migration,M090000US001,M09,Franklin/Delaware county region,2050,Y-JUN,2030658,1990641,...,1305278,331485,3569,23828,12620,824348,884052,1179220,1128747,1405071


Write the data to disk.

In [579]:
outputWide.to_csv(OUTPUT_WIDE_PATH, index=False)

Create a resource file and validate the data.

In [580]:
outputWideResource = morpc.frictionless.create_resource(OUTPUT_WIDE_FILENAME, 
    resourcePath=OUTPUT_WIDE_RESOURCE_PATH,
    title=f"MORPC County Control Totals (wide format)", 
    description= f"Every four years, MORPC produces a set of county-level projections for several variables in support of our " +
            f"transportation planning work. These are known collectively as the 'county control totals'. This dataset may include " +
            f"several vintages of projections. Projections are included for counties in the MORPC 15-county region of interest, which " +
            f"includes the follow counties: {', '.join(morpc.CONST_REGIONS['REGION15'])}. " +
            f"Additionally, projections are included for the 15-county region as a whole and several other county-based regions " +
            f"commonly used to describe Central Ohio. More information about these projections and the methods used to create " +
            f"them is available in the workflow documentation available at https://github.com/morpc/morpc-county-controls. " +
            f"For inquiries related to these projections, please send an email to dataandmaps@morpc.org.",
    writeResource=True,
    validate=True
)
outputWideResource

2026-08-27 22:02:38,649 | INFO | morpc.frictionless.frictionless.create_resource: Format not specified. Using format derived from data file extension: csv
2026-08-27 22:02:38,650 | INFO | morpc.frictionless.frictionless.create_resource: morpc.frictionless.create_resource | INFO | Schema path not specified. Using path derived from data file path: morpc-county-controls-wide.schema.yaml
2026-08-27 22:02:38,651 | INFO | morpc.frictionless.frictionless.create_resource: Name not specified. Using placeholder value derived from data filename: morpc-county-controls-wide
2026-08-27 22:02:38,652 | INFO | morpc.frictionless.frictionless.create_resource: Sources not specified. No source information will be included in the resource.
2026-08-27 22:02:38,655 | INFO | morpc.frictionless.frictionless.create_resource: Writing Frictionless Resource file to output_data\morpc-county-controls-wide.resource.yaml
2026-08-27 22:02:38,689 | INFO | morpc.frictionless.frictionless.create_resource: Validating resou

{'name': 'morpc-county-controls-wide',
 'type': 'table',
 'title': 'MORPC County Control Totals (wide format)',
 'description': 'Every four years, MORPC produces a set of county-level '
                'projections for several variables in support of our '
                'transportation planning work. These are known collectively as '
                "the 'county control totals'. This dataset may include several "
                'vintages of projections. Projections are included for '
                'counties in the MORPC 15-county region of interest, which '
                'includes the follow counties: Delaware, Fairfield, Fayette, '
                'Franklin, Hocking, Knox, Licking, Logan, Madison, Marion, '
                'Morrow, Perry, Pickaway, Ross, Union. Additionally, '
                'projections are included for the 15-county region as a whole '
                'and several other county-based regions commonly used to '
                'describe Central Ohio. More info

### Long form table

Load the output schema.

Create a working copy of the county controls data and reset the index.

Melt the data into long format.

Add a columns with human-readable variable names.

Extract only the fields required by the schema and ensure they are in the correct order.

Ensure that all fields are cast as the data types required by the schema.

Sort the records in a logical order.

Inspect the data.

Write the data to disk.

Create a resource file and validate the data.

## Export notebook as HTML

In [ ]:
if(EXPORT_NOTEBOOK_AS_HTML):
    logger.info("Exporting notebook to HTML format")    
    morpc.notebook_to_html()
else:
    logger.info("Skipping export to HTML format")    

## Produce charts (move to -profile?)